Dataset source: [Kaggle Link](https://www.kaggle.com/datasets/jealousleopard/goodreadsbooks/data)


### Pre-exploration notes:
- Goodreads withdrew their API access, so this sample is quite outdated and misses releases from the past 6 years
- There seems to be a mix between authors, translators and even illustrators in the same author column string, needs to be split and verified who is authored (potentially some type of scraping tool or AI integration for this?)
- Same book can be duplicated in different editions/publications or different translations - this is skewing ratings, should bperhaps be grouped
- To not lose information about publishers, it would be best to seperate tables with many to many relationships
- ISBN seems to be what could reliably connect this datasets with different ones (ideas: Booker Prize, the bookdepository dataset etc.)
- What would be the most reliable way to identify the same book across different languages and editions? (For example to compare rating or page count in different languages)


### Imports:

In [ ]:
from bookstats.config import FILTERED_DATA, PUBLISHERS
from bookstats.formatting import clean_headers
import numpy as np
import pandas as pd

df = pd.read_csv(FILTERED_DATA)
clean_headers(df)

#### Shape and Basic Information:

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.isna().sum()

Note: No null values

---

#### Publishers:

Looking at how many publishers we're dealing with, exporting as separate csv to have a quick look and material for later (likely its own table after normalisation).

In [ ]:
print(f"Number of distinct publishers: {df['publisher'].nunique()}")
publishers = df['publisher'].drop_duplicates().sort_values()

publishers.to_csv(PUBLISHERS, index=False)


Whitespace errors (double space, missing space, " : ") need to be addressed in cleaning scripts. 

**Questions arising:**
- What should count as distinct imprint?
- How to validate official publisher name and group similar? (Scraping? ISBN as source?)
- How to automate the process without researching every single publisher? (Example: Atheneum Books and Atheneum Books for Young Readers -> two imprints?)

---

#### Missing values:

- Numerics with 0
- Empty strings
- unknowns

In [ ]:
print(df["num_pages"].min())
print(df["average_rating"].min())
print(df["ratings_count"].min())
print(df["text_reviews_count"].min())

In [ ]:
(df[["num_pages", "average_rating","ratings_count", "text_reviews_count"]] == 0).sum()

Note: Zero values revealed across all numeric columns. Unrated books are conceptually ok, 0 page count not.

In [ ]:
df[(df["average_rating"] == 0) & (df["ratings_count"] == 0)].count()

Note: There is an overlap, all 0 ratings_count books also have 0 average.

In [ ]:
df[(df["average_rating"] != 0) & (df["ratings_count"] == 0)]

Note: There are books with 0 ratings yet still a calculated average -> needs to be converted to NaN.

In [ ]:
df[df["num_pages"] == 0].head(5)

In [ ]:
print(df["title"].value_counts().head(3))
print(df["authors"].value_counts().head(3))
print(df["publisher"].value_counts().head(3))
print(df["isbn"].value_counts().head(3))
print(df["isbn13"].value_counts().head(3))

In [ ]:
df[["title", "authors", "publisher", "isbn", "isbn13"]].isin(["", "Unknown", "unknown"]).sum()

---

## Initial notes
If I want to expand, I will need to use a scraping tool to gather the following:

- Genres for interesting stats
- Profession connected the name (author, translator, illustrator -> not differenciated atm)
- Gender/nationality of author? Could be interesting for some stats
- Awards


**Cleaning and processing needs**

- Keeping data for English language only
- Grouping the same book or same publisher into one
- Separating author names and assigning profession
- Remove box sets and keep logic being 1 row per edition